# Documentation of convert EU field boundaries to Pmtiles for web hosting services
We processed EU field boundaries from Planet into PMTiles to support Web GIS services such as EcodataCube(https://ecodatacube.eu).

- EU field boundaries data can be accessed via Zenodo: https://doi.org/10.5281/zenodo.14229033
- EU crop mapping on field boundaries via Zenodo: https://doi.org/10.5281/zenodo.14842659

In [ ]:
# doownload crop boundaries from Zenodo
!wget -nc https://zenodo.org/records/14229033/files/field_boundaries.parquet?download=1
!wget -nc https://zenodo.org/records/14842659/files/field_boundaries_crop_classification.csv?download=1

In [ ]:
from shapely.geometry import Polygon, Point
import pandas as pd
import os
os.environ['USE_PYGEOS'] = '0'
import geopandas as gpd
filed_bounaries = pd.read_parquet('field_boundaries.parquet?download=1')

In [ ]:
df_crop=pd.read_csv('field_boundaries_crop_classification.csv?download=1')

In [ ]:
filed_bounaries['id']=filed_bounaries['id'].astype(int)

In [ ]:
filed_bounaries.head().set_index('id')

In [ ]:
df_join=filed_bounaries.set_index('id').join(df_crop.rename(columns={'field_boundary_id':'id'}).set_index('id'), on='id')

In [ ]:
del filed_bounaries, df_crop

In [ ]:
from shapely import from_wkb

In [ ]:
df_join['geometry']=df_join.geometry.apply(lambda x: from_wkb(x))

In [ ]:
gpd_filed_bounaries=df_join.set_geometry('geometry')

In [ ]:
del df_join

In [ ]:
df_join=None

In [ ]:
gpd_filed_bounaries.to_parquet('field_boundaries_crop_classification.parquet')

In [ ]:
gpd_filed_bounaries.head().columns

In [ ]:
gpd_filed_bounaries['area']=gpd_filed_bounaries['area'].astype('float32')

In [ ]:
gpd_filed_bounaries.to_file('field_boundaries_crop_classification.geojson',Driver='GeoJSON')

In [ ]:
# tippecanoe now support converting from GeoJSON
! tippecanoe  -o field_boundaries_crop_classification.pmtiles --force --maximum-tile-bytes=1000000 --no-feature-limit --read-parallel  --hilbert --detect-shared-borders --drop-densest-as-needed --extend-zooms-if-still-dropping --full-detail=15  --low-detail=12 --minimum-zoom=0 --maximum-zoom=15 field_boundaries_crop_classification.geojson